# NTE lookup-key normalization benchmark

This benchmark tests normalization for the initial entity-resolution key.

MCN provides an attested name plus the expected GeoNames ID. Each MCN name is treated as lookup input.
A policy succeeds when the normalized key resolves back to the expected entity.

Policies are tested incrementally:

1. `exact`
2. `lower`
3. `casefold` — NFC + whitespace cleanup + Unicode casefold
4. `diacritic_fold` — casefold + NFD + removal of combining marks

Two further tiers run only on the rows the ladder can't resolve (see *Romanized keys*):

5. `compact` — diacritic fold, ignoring spaces, punctuation and modifier letters
6. `romanized` — the compact key of every NTE name after NTE's `Romanizer`

A last section measures how close NTE's romanization comes to MCN's spelling for names that NTE stores in a non-Latin script.

Normalization is computed at runtime.

In [ ]:
import sqlite3
import sys
import time
import difflib
import statistics
import collections
import unicodedata
import pandas as pd
import xml.etree.ElementTree as ET

from pathlib import Path
from dataclasses import dataclass
from enum import Enum

pd.set_option("display.max_rows", 1000)
pd.set_option("display.max_columns", 80)
pd.set_option("display.max_colwidth", 100)

PROJECT_ROOT  = Path.cwd().parent
MCN_PATH      = PROJECT_ROOT / "data" / "repos" / "more-cultural-names"
MCN_LOCATIONS = MCN_PATH / "locations.xml"
MCN_LANGUAGES = MCN_PATH / "languages.xml"
DB_PATH       = PROJECT_ROOT / "data" / "names.sqlite"

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from name_transduction_engine.datasets.language_codes.data_provision import read_registry
from name_transduction_engine.normalization.language_code_normalization import (
    UnknownLanguageError,
    resolve_user_language,
)
from name_transduction_engine.transliteration.romanization import Romanizer

In [ ]:
def _clean_ws(text: str) -> str:
    return " ".join(text.strip().split())

def norm_lower(text: str) -> str:
    return _clean_ws(unicodedata.normalize("NFC", text)).lower()

def norm_casefold(text: str) -> str:
    return _clean_ws(unicodedata.normalize("NFC", text)).casefold()

def norm_diacritic_fold(text: str) -> str:
    folded = norm_casefold(text)
    decomposed = unicodedata.normalize("NFD", folded)
    stripped = "".join(ch for ch in decomposed if not unicodedata.category(ch).startswith("M"))
    return unicodedata.normalize("NFC", stripped)

def get_conn() -> sqlite3.Connection:
    conn = sqlite3.connect(DB_PATH)
    conn.row_factory = sqlite3.Row
    conn.execute("PRAGMA foreign_keys = ON;")
    conn.create_function("nte_lower", 1, norm_lower, deterministic=True)
    conn.create_function("nte_casefold", 1, norm_casefold, deterministic=True)
    conn.create_function("nte_diacritic_fold", 1, norm_diacritic_fold, deterministic=True)
    return conn

def run_query(sql: str, params: tuple = ()) -> pd.DataFrame:
    with get_conn() as conn:
        return pd.read_sql_query(sql, conn, params=params)

samples = ["Istanbul", "İstanbul", "Málaga", "MALAGA", "  New   York  ", "Straße"]
display(pd.DataFrame({
    "raw": samples,
    "lower": [norm_lower(x) for x in samples],
    "casefold": [norm_casefold(x) for x in samples],
    "diacritic_fold": [norm_diacritic_fold(x) for x in samples],
}))


## Load MCN benchmark data

This mirrors the general benchmark. The GeoNames ID attached to each MCN name is treated as the expected
resolved entity.


In [ ]:
languages_tree = ET.parse(MCN_LANGUAGES)

@dataclass(frozen=True)
class LangInfo:
    mcn_id: str
    iso1: str | None
    iso2: str | None
    iso3: str | None

    @property
    def iso_codes(self) -> tuple[str, ...]:
        return tuple(dict.fromkeys(c for c in (self.iso1, self.iso2, self.iso3) if c))

lang_info = {}
for lang in languages_tree.getroot().findall("Language"):
    mcn_id = (lang.findtext("Id") or "").strip()
    if not mcn_id:
        continue
    node = lang.find("Code")
    iso1 = iso2 = iso3 = None
    if node is not None:
        iso1 = node.attrib.get("iso-639-1") or None
        iso2 = node.attrib.get("iso-639-2") or None
        iso3 = node.attrib.get("iso-639-3") or None
    lang_info[mcn_id] = LangInfo(mcn_id, iso1, iso2, iso3)

locations_tree = ET.parse(MCN_LOCATIONS)
ck3_names = {}

for loc in locations_tree.iter("LocationEntity"):
    geonames_tag = loc.find("GeoNamesId")
    if geonames_tag is None:
        continue

    geonames_id = int(geonames_tag.text)
    names = {n.attrib["language"]: n.attrib["value"] for n in loc.findall("Names/Name")}
    if not names:
        continue

    for game_id in loc.findall("GameIds/GameId"):
        if game_id.attrib.get("game") != "CK3":
            continue
        title = game_id.text.strip() if game_id.text is not None else ""
        if title.startswith("d_nf") or title.startswith("b_"):
            continue
        ck3_names[(title, geonames_id)] = names

attested_rows = [
    (title, gid, lang, name)
    for (title, gid), names in ck3_names.items()
    for lang, name in names.items()
]

ck3_geonames_ids = sorted({gid for _, gid in ck3_names})

print(f"CK3 title/entity pairs : {len(ck3_names)}")
print(f"distinct entities      : {len(ck3_geonames_ids)}")
print(f"attested input rows    : {len(attested_rows)}")


## Cache NTE names for the benchmark entities

For initial lookup resolution, the candidate surfaces include:

- `geoname.name`
- `alternate_name.alternate_name`
- linked `wikidata_location_name.name`

SQLite computes the experimental normalization keys at runtime. Each surface also carries its normalized
language (`lang`), which the romanization section uses, and the normalized key stored at load time
(`stored_norm`), which the audit below checks.

In [ ]:
@dataclass(frozen=True)
class NameSurface:
    source: str
    surface_type: str
    geonames_id: int
    name: str
    lang: str | None          # registry code; None for primary names and untagged rows
    lower_key: str
    casefold_key: str
    diacritic_key: str
    stored_norm: str | None   # normalized key stored at load time

def get_nte_surfaces_for_geonames_id(gid: int) -> list[NameSurface]:
    df = run_query("""
        SELECT
            'geonames' AS source,
            'primary' AS surface_type,
            g.geonameid AS geonames_id,
            g.name AS name,
            NULL AS lang,
            nte_lower(g.name) AS lower_key,
            nte_casefold(g.name) AS casefold_key,
            nte_diacritic_fold(g.name) AS diacritic_key,
            g.normalized_name AS stored_norm
        FROM geoname AS g
        WHERE g.geonameid = ?

        UNION ALL

        SELECT
            'geonames' AS source,
            'alternate' AS surface_type,
            a.geonameid AS geonames_id,
            a.alternate_name AS name,
            a.lang AS lang,
            nte_lower(a.alternate_name) AS lower_key,
            nte_casefold(a.alternate_name) AS casefold_key,
            nte_diacritic_fold(a.alternate_name) AS diacritic_key,
            a.normalized_name AS stored_norm
        FROM alternate_name AS a
        WHERE a.geonameid = ?

        UNION ALL

        SELECT
            'wikidata' AS source,
            n.term_type AS surface_type,
            g.geonames_id AS geonames_id,
            n.name AS name,
            n.lang AS lang,
            nte_lower(n.name) AS lower_key,
            nte_casefold(n.name) AS casefold_key,
            nte_diacritic_fold(n.name) AS diacritic_key,
            n.normalized_name AS stored_norm
        FROM wikidata_location_geonames AS g
        JOIN wikidata_location_name AS n ON n.qid = g.qid
        WHERE g.geonames_id = ?
    """, (gid, gid, gid))

    return [
        NameSurface(
            row.source,
            row.surface_type,
            int(row.geonames_id),
            row.name,
            None if pd.isna(row.lang) else row.lang,
            row.lower_key,
            row.casefold_key,
            row.diacritic_key,
            None if pd.isna(row.stored_norm) else row.stored_norm,
        )
        for row in df.itertuples(index=False)
    ]

nte_cache = {gid: get_nte_surfaces_for_geonames_id(gid) for gid in ck3_geonames_ids}

print(f"entities cached     : {len(nte_cache)}")
print(f"total name surfaces : {sum(len(v) for v in nte_cache.values())}")

## Build resolution indexes

Each normalization key maps to all benchmark GeoNames entities carrying that key. This lets the benchmark
measure both recall and ambiguity.


In [ ]:
def build_index(key_fn):
    entity_index = collections.defaultdict(set)
    surface_index = collections.defaultdict(list)

    for gid, surfaces in nte_cache.items():
        for s in surfaces:
            key = key_fn(s)
            entity_index[key].add(gid)
            surface_index[key].append(s)

    return dict(entity_index), dict(surface_index)

exact_index, exact_surfaces = build_index(lambda s: s.name)
lower_index, lower_surfaces = build_index(lambda s: s.lower_key)
casefold_index, casefold_surfaces = build_index(lambda s: s.casefold_key)
diacritic_index, diacritic_surfaces = build_index(lambda s: s.diacritic_key)

print("distinct keys:")
print(f"  exact          {len(exact_index)}")
print(f"  lower          {len(lower_index)}")
print(f"  casefold       {len(casefold_index)}")
print(f"  diacritic_fold {len(diacritic_index)}")


In [ ]:
class MatchStatus(Enum):
    EXACT = "exact"
    LOWER = "lower"
    CASEFOLD = "casefold"
    DIACRITIC_FOLD = "diacritic_fold"
    WRONG_ONLY = "wrong_only"
    NO_MATCH = "no_match"

@dataclass
class Result:
    title: str
    geonames_id: int
    mcn_lang: str
    input_name: str
    status: MatchStatus
    matched_entity_ids: tuple[int, ...]
    matched_names: tuple[str, ...]
    matched_sources: tuple[str, ...]
    ambiguous: bool

def compare_one(title, gid, mcn_lang, input_name):
    ladder = [
        (MatchStatus.EXACT, input_name, exact_index, exact_surfaces),
        (MatchStatus.LOWER, norm_lower(input_name), lower_index, lower_surfaces),
        (MatchStatus.CASEFOLD, norm_casefold(input_name), casefold_index, casefold_surfaces),
        (MatchStatus.DIACRITIC_FOLD, norm_diacritic_fold(input_name), diacritic_index, diacritic_surfaces),
    ]

    for status, key, idx, surfaces in ladder:
        ids = tuple(sorted(idx.get(key, set())))
        if gid in ids:
            rows = [s for s in surfaces.get(key, []) if s.geonames_id == gid]
            return Result(
                title, gid, mcn_lang, input_name, status, ids,
                tuple(dict.fromkeys(s.name for s in rows)),
                tuple(sorted({s.source for s in rows})),
                len(ids) > 1,
            )

    broad_key = norm_diacritic_fold(input_name)
    ids = tuple(sorted(diacritic_index.get(broad_key, set())))
    status = MatchStatus.WRONG_ONLY if ids else MatchStatus.NO_MATCH
    return Result(title, gid, mcn_lang, input_name, status, ids, (), (), False)

results = [compare_one(*row) for row in attested_rows]
print(f"comparisons run: {len(results)}")


# Global results

The status counts are incremental, so `diacritic_fold` means that all less aggressive policies failed first.


In [ ]:
counts = collections.Counter(r.status for r in results)
n = len(results)

for status in MatchStatus:
    c = counts[status]
    print(f"{status.value:16s} {c:7d} ({c/n:.2%})")

resolved_tiers = [
    MatchStatus.EXACT,
    MatchStatus.LOWER,
    MatchStatus.CASEFOLD,
    MatchStatus.DIACRITIC_FOLD,
]

cum = 0
print()
for status in resolved_tiers:
    cum += counts[status]
    print(f"resolved through {status.value:16s}: {cum:7d}/{n} = {cum/n:.2%}")

before_diacritic = counts[MatchStatus.DIACRITIC_FOLD] + counts[MatchStatus.WRONG_ONLY] + counts[MatchStatus.NO_MATCH]
if before_diacritic:
    rescued = counts[MatchStatus.DIACRITIC_FOLD]
    print()
    print(f"diacritic-fold rescue among prior misses: {rescued}/{before_diacritic} = {rescued/before_diacritic:.2%}")


## Ambiguity cost


In [ ]:
rows = []
for status in resolved_tiers:
    subset = [r for r in results if r.status == status]
    ambiguous = sum(r.ambiguous for r in subset)
    rows.append({
        "tier": status.value,
        "resolved_rows": len(subset),
        "ambiguous_rows": ambiguous,
        "ambiguity_rate": ambiguous / len(subset) if subset else 0.0,
        "mean_entities": (
            sum(len(r.matched_entity_ids) for r in subset) / len(subset)
            if subset else 0.0
        ),
    })

ambiguity_df = pd.DataFrame(rows)
display(ambiguity_df.style.format({
    "ambiguity_rate": "{:.2%}",
    "mean_entities": "{:.3f}",
}))


## Examples rescued by each normalization tier


In [ ]:
def examples(status, limit=100):
    return pd.DataFrame([
        {
            "title": r.title,
            "geonames_id": r.geonames_id,
            "language": r.mcn_lang,
            "input": r.input_name,
            "matched_db_names": list(r.matched_names),
            "sources": list(r.matched_sources),
            "entity_count": len(r.matched_entity_ids),
        }
        for r in results if r.status == status
    ]).head(limit)

print("lower:")
display(examples(MatchStatus.LOWER))

print("casefold:")
display(examples(MatchStatus.CASEFOLD))

print("diacritic_fold:")
display(examples(MatchStatus.DIACRITIC_FOLD))


## Per-language gains


In [ ]:
by_lang = collections.defaultdict(list)
for r in results:
    by_lang[r.mcn_lang].append(r)

rows = []
for lang, rs in by_lang.items():
    c = collections.Counter(r.status for r in rs)
    attested = len(rs)
    resolved = sum(c[s] for s in resolved_tiers)
    rows.append({
        "mcn_lang": lang,
        "attested": attested,
        "exact": c[MatchStatus.EXACT],
        "lower_gain": c[MatchStatus.LOWER],
        "casefold_gain": c[MatchStatus.CASEFOLD],
        "diacritic_gain": c[MatchStatus.DIACRITIC_FOLD],
        "resolved_rate": resolved / attested if attested else 0.0,
    })

per_lang_df = pd.DataFrame(rows).sort_values(
    ["diacritic_gain", "attested"], ascending=[False, False]
).reset_index(drop=True)

display(per_lang_df.head(100).style.format({"resolved_rate": "{:.1%}"}))


## Collision rate inside the CK3 benchmark universe

This is not a full-database collision scan. It is intentionally limited to the entities already present in
the benchmark, so it remains cheap while still showing whether a normalization policy collapses many distinct
place names together.


In [ ]:
def collision_summary(index, label):
    ambiguous = [ids for ids in index.values() if len(ids) > 1]
    return {
        "policy": label,
        "distinct_keys": len(index),
        "ambiguous_keys": len(ambiguous),
        "ambiguous_key_rate": len(ambiguous) / len(index) if index else 0.0,
        "max_entities_per_key": max((len(ids) for ids in index.values()), default=0),
    }

collision_df = pd.DataFrame([
    collision_summary(exact_index, "exact"),
    collision_summary(lower_index, "lower"),
    collision_summary(casefold_index, "casefold"),
    collision_summary(diacritic_index, "diacritic_fold"),
])

display(collision_df.style.format({"ambiguous_key_rate": "{:.2%}"}))

collision_examples = []
for key, ids in diacritic_index.items():
    if len(ids) <= 1:
        continue
    collision_examples.append({
        "key": key,
        "entity_count": len(ids),
        "entity_ids": sorted(ids),
        "raw_names": sorted({s.name for s in diacritic_surfaces[key]})[:30],
    })

display(
    pd.DataFrame(collision_examples)
    .sort_values(["entity_count", "key"], ascending=[False, True])
    .head(100)
)


## Compare against the normalized fields already stored

`geoname.normalized_name`, `alternate_name.normalized_name` and `wikidata_location_name.normalized_name` are all computed at load time by `normalize_name` (for Wikidata, recomputed from the raw label). They are compared here against the experimental runtime keys. `equals_casefold` should be 100% for every source. A lower number means the stored key and the runtime `casefold` policy have drifted apart.

In [ ]:
stored_rows = [
    s
    for surfaces in nte_cache.values()
    for s in surfaces
    if s.stored_norm is not None
]

audit = []
for source, surface_types in (
    ("geonames", ("primary",)),
    ("geonames", ("alternate",)),
    ("wikidata", None),
):
    rows = [
        s for s in stored_rows
        if s.source == source
        and (surface_types is None or s.surface_type in surface_types)
        and s.stored_norm != ""
    ]
    n_rows = len(rows)
    if not n_rows:
        continue

    audit.append({
        "source": source if surface_types is None else f"{source}:{surface_types[0]}",
        "rows": n_rows,
        "equals_lower": sum(s.stored_norm == s.lower_key for s in rows) / n_rows,
        "equals_casefold": sum(s.stored_norm == s.casefold_key for s in rows) / n_rows,
        "equals_diacritic_fold": sum(s.stored_norm == s.diacritic_key for s in rows) / n_rows,
    })

display(pd.DataFrame(audit).style.format({
    "equals_lower": "{:.1%}",
    "equals_casefold": "{:.1%}",
    "equals_diacritic_fold": "{:.1%}",
}))

## Wrong-only broad matches


In [ ]:
wrong_only_df = pd.DataFrame([
    {
        "title": r.title,
        "expected_geonames_id": r.geonames_id,
        "language": r.mcn_lang,
        "input": r.input_name,
        "diacritic_key": norm_diacritic_fold(r.input_name),
        "resolved_entity_ids": list(r.matched_entity_ids),
    }
    for r in results if r.status == MatchStatus.WRONG_ONLY
])

print(f"wrong-only rows: {len(wrong_only_df)}")
display(wrong_only_df.head(200))


# Romanized keys

Many MCN names are written in Latin script even when the language isn't: Russian "Kazan'", Chinese "Beijing". NTE stores those names in their native script, so none of the four tiers above can resolve them.

This section adds two tiers for the rows the ladder left unresolved (`wrong_only` or `no_match`):

- **compact**: the diacritic-folded key with spaces, punctuation and modifier letters (ʹ, ʼ) removed as well. This separates rescues caused by apostrophes and word breaks from real script differences.
- **romanized**: the compact key of every NTE name after running it through NTE's `Romanizer` (the technical, ISO 9–style one). An MCN input resolves here if its compact key equals the compact key of a romanized NTE name for the expected entity.

Both tiers are looser than any in the ladder, so both report their ambiguity cost in the benchmark universe. A full-database collision audit would mean romanizing every name in the database, so it isn't run here.

In [ ]:
romanizer = Romanizer()
_rom_cache: dict[str, object] = {}

def romanize(text: str):
    """Cached: the same name occurs under many tags and entities"""
    r = _rom_cache.get(text)
    if r is None:
        r = romanizer.romanize(text)
        _rom_cache[text] = r
    return r

def compact_key(text: str) -> str:
    """Diacritic fold, then keep letters and digits only; drop modifier letters (ʹ ʼ)"""
    folded = norm_diacritic_fold(text)
    return "".join(
        ch for ch in folded
        if ch.isalnum() and unicodedata.category(ch) != "Lm"
    )

def is_latin_text(text: str) -> bool:
    letters = [ch for ch in text if ch.isalpha()]
    return bool(letters) and all("LATIN" in unicodedata.name(ch, "") for ch in letters)

t0 = time.perf_counter()
all_surfaces = [s for surfaces in nte_cache.values() for s in surfaces]
for s in all_surfaces:
    romanize(s.name)
print(f"romanized {len(_rom_cache):,} distinct NTE names in {time.perf_counter() - t0:.1f}s")

transform_counts = collections.Counter(romanize(s.name).transform for s in all_surfaces)
print("\nTransforms used (name surfaces):")
for transform, count in transform_counts.most_common():
    print(f"  {transform:22s} {count:7d}")

In [ ]:
compact_index, compact_surfaces = build_index(lambda s: compact_key(s.name))
roman_index, roman_surfaces = build_index(lambda s: compact_key(romanize(s.name).text))

class ExtraStatus(Enum):
    COMPACT    = "compact"
    ROMANIZED  = "romanized"
    WRONG_ONLY = "wrong_only"
    NO_MATCH   = "no_match"

@dataclass
class ExtraResult:
    base:        Result
    status:      ExtraStatus
    key:         str
    entity_ids:  tuple[int, ...]
    matched:     tuple[str, ...]    # NTE names (native) that produced the match
    romanized:   tuple[str, ...]    # their romanized forms (romanized tier only)

def extra_one(r: Result) -> ExtraResult:
    key = compact_key(r.input_name)
    ids = tuple(sorted(compact_index.get(key, set())))
    if key and r.geonames_id in ids:
        rows = [s for s in compact_surfaces[key] if s.geonames_id == r.geonames_id]
        return ExtraResult(r, ExtraStatus.COMPACT, key, ids,
                           tuple(dict.fromkeys(s.name for s in rows)), ())

    rkey = compact_key(romanize(r.input_name).text)
    rids = tuple(sorted(roman_index.get(rkey, set())))
    if rkey and r.geonames_id in rids:
        rows = [s for s in roman_surfaces[rkey] if s.geonames_id == r.geonames_id]
        return ExtraResult(r, ExtraStatus.ROMANIZED, rkey, rids,
                           tuple(dict.fromkeys(s.name for s in rows)),
                           tuple(dict.fromkeys(romanize(s.name).text for s in rows)))

    any_ids = tuple(sorted(set(ids) | set(rids)))
    status = ExtraStatus.WRONG_ONLY if any_ids else ExtraStatus.NO_MATCH
    return ExtraResult(r, status, rkey, any_ids, (), ())

n_rows_total = len(results)
unresolved = [r for r in results if r.status in (MatchStatus.WRONG_ONLY, MatchStatus.NO_MATCH)]
extra_results = [extra_one(r) for r in unresolved]
extra_counts = collections.Counter(e.status for e in extra_results)

print(f"rows unresolved by the four-tier ladder: {len(unresolved)} of {n_rows_total} "
      f"({len(unresolved)/n_rows_total:.2%})")
for status in ExtraStatus:
    c = extra_counts[status]
    print(f"  {status.value:11s} {c:6d}  ({c/len(unresolved):.2%} of unresolved)" if unresolved else "")

cum = n_rows_total - len(unresolved)
print()
print(f"resolved through diacritic_fold: {cum}/{n_rows_total} = {cum/n_rows_total:.2%}")
for status in (ExtraStatus.COMPACT, ExtraStatus.ROMANIZED):
    cum += extra_counts[status]
    print(f"resolved through {status.value:14s}: {cum}/{n_rows_total} = {cum/n_rows_total:.2%}")

rows = []
for status in (ExtraStatus.COMPACT, ExtraStatus.ROMANIZED):
    subset = [e for e in extra_results if e.status == status]
    rows.append({
        "tier": status.value,
        "rescued_rows": len(subset),
        "ambiguous_rows": sum(len(e.entity_ids) > 1 for e in subset),
        "ambiguity_rate": (sum(len(e.entity_ids) > 1 for e in subset) / len(subset)) if subset else 0.0,
        "mean_entities": (sum(len(e.entity_ids) for e in subset) / len(subset)) if subset else 0.0,
    })
print("\nAmbiguity cost of the extra tiers:")
display(pd.DataFrame(rows).style.format({"ambiguity_rate": "{:.2%}", "mean_entities": "{:.3f}"}))

print("Collision rate in the benchmark universe, compared with the ladder's loosest key:")
display(pd.DataFrame([
    collision_summary(diacritic_index, "diacritic_fold"),
    collision_summary(compact_index, "compact"),
    collision_summary(roman_index, "romanized"),
]).style.format({"ambiguous_key_rate": "{:.2%}"}))

In [ ]:
by_lang_extra = collections.defaultdict(collections.Counter)
for e in extra_results:
    by_lang_extra[e.base.mcn_lang][e.status] += 1

lang_rows = []
for lang, c in by_lang_extra.items():
    lang_rows.append({
        "mcn_lang": lang,
        "unresolved": sum(c.values()),
        "compact": c[ExtraStatus.COMPACT],
        "romanized": c[ExtraStatus.ROMANIZED],
        "still_unresolved": c[ExtraStatus.WRONG_ONLY] + c[ExtraStatus.NO_MATCH],
    })
lang_extra_df = (
    pd.DataFrame(lang_rows)
    .sort_values(["romanized", "compact", "unresolved"], ascending=False)
    .reset_index(drop=True)
)
print("Rescues per MCN language:")
display(lang_extra_df.head(40))

def extra_examples(status, limit=60):
    return pd.DataFrame([
        {
            "language": e.base.mcn_lang,
            "input": e.base.input_name,
            "key": e.key,
            "nte_names": list(e.matched[:3]),
            "romanized_as": list(e.romanized[:3]),
            "entity_count": len(e.entity_ids),
        }
        for e in extra_results if e.status == status
    ]).head(limit)

print("compact rescues:")
display(extra_examples(ExtraStatus.COMPACT))
print("romanized rescues:")
display(extra_examples(ExtraStatus.ROMANIZED))

# Romanization quality against MCN

For **display**, not resolution: how close does NTE's romanization come to the spelling MCN uses?

This compares rows where:
- MCN's name is in Latin script;
- the MCN language resolves to a registry code;
- NTE has at least one name for the expected entity tagged with that code and written in a non-Latin script.

Each such NTE name is romanized, and the closest one to MCN's name is kept. Levels of agreement:

- **exact**: identical strings.
- **fold**: equal after case and diacritic folding.
- **compact**: equal after also ignoring spaces, punctuation and modifier letters.
- **similarity**: 1 − (edit distance ÷ longer length), after case folding only, so diacritics still count as differences.

The romanizer is technical by design (reversible, ISO 9–style), so low exact agreement is expected. The useful output is *how far* it is from MCN's spelling, and the character substitutions that explain the difference.

In [ ]:
with get_conn() as conn:
    registry = read_registry(conn)

def resolve_mcn_code(li: LangInfo) -> str | None:
    for raw in (li.iso1, li.iso3, li.iso2):
        if not raw:
            continue
        try:
            return resolve_user_language(raw, registry).tag.lang
        except UnknownLanguageError:
            continue
    return None

mcn_code = {mcn_id: resolve_mcn_code(li) for mcn_id, li in lang_info.items()}

def levenshtein(a: str, b: str) -> int:
    if len(a) < len(b):
        a, b = b, a
    previous = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        current = [i]
        for j, cb in enumerate(b, 1):
            current.append(min(previous[j] + 1, current[j - 1] + 1, previous[j - 1] + (ca != cb)))
        previous = current
    return previous[-1]

def similarity(a: str, b: str) -> float:
    longest = max(len(a), len(b))
    return 1.0 - levenshtein(a, b) / longest if longest else 1.0

@dataclass
class QualityRow:
    title:       str
    geonames_id: int
    mcn_lang:    str
    code:        str
    mcn_name:    str
    native:      str      # NTE's native-script name that romanized closest
    romanized:   str
    transform:   str
    confidence:  str
    exact:       bool
    fold:        bool
    compact:     bool
    sim:         float    # case-folded, diacritics kept
    sim_fold:    float    # diacritics folded too

quality_rows: list[QualityRow] = []
for r in results:
    code = mcn_code.get(r.mcn_lang)
    if code is None or not is_latin_text(r.input_name):
        continue
    natives = sorted({
        s.name for s in nte_cache.get(r.geonames_id, [])
        if s.lang == code and romanize(s.name).transform != "identity"
    })
    if not natives:
        continue

    target = norm_casefold(r.input_name)
    best = max(natives, key=lambda nat: (similarity(target, norm_casefold(romanize(nat).text)), nat))
    rom = romanize(best)
    quality_rows.append(QualityRow(
        r.title, r.geonames_id, r.mcn_lang, code, r.input_name, best, rom.text,
        rom.transform, rom.confidence,
        exact=rom.text == r.input_name,
        fold=norm_diacritic_fold(rom.text) == norm_diacritic_fold(r.input_name),
        compact=compact_key(rom.text) == compact_key(r.input_name),
        sim=similarity(target, norm_casefold(rom.text)),
        sim_fold=similarity(norm_diacritic_fold(r.input_name), norm_diacritic_fold(rom.text)),
    ))

nq = len(quality_rows)
print(f"comparable rows (Latin MCN name vs non-Latin NTE name, same language): {nq}")
if nq:
    print(f"  exact    {sum(q.exact for q in quality_rows):6d}  ({sum(q.exact for q in quality_rows)/nq:.1%})")
    print(f"  fold     {sum(q.fold for q in quality_rows):6d}  ({sum(q.fold for q in quality_rows)/nq:.1%})")
    print(f"  compact  {sum(q.compact for q in quality_rows):6d}  ({sum(q.compact for q in quality_rows)/nq:.1%})")
    print(f"  mean similarity (diacritics kept)  : {statistics.mean(q.sim for q in quality_rows):.3f}")
    print(f"  mean similarity (diacritics folded): {statistics.mean(q.sim_fold for q in quality_rows):.3f}")

quality_by_lang = []
grouped: dict[str, list[QualityRow]] = collections.defaultdict(list)
for q in quality_rows:
    grouped[q.mcn_lang].append(q)
for lang, qs in grouped.items():
    transforms = collections.Counter(q.transform for q in qs)
    quality_by_lang.append({
        "mcn_lang":        lang,
        "code":            qs[0].code,
        "rows":            len(qs),
        "exact":           sum(q.exact for q in qs) / len(qs),
        "fold":            sum(q.fold for q in qs) / len(qs),
        "compact":         sum(q.compact for q in qs) / len(qs),
        "sim":             statistics.mean(q.sim for q in qs),
        "sim_fold":        statistics.mean(q.sim_fold for q in qs),
        "low_confidence":  sum(q.confidence == "low" for q in qs) / len(qs),
        "transform":       transforms.most_common(1)[0][0],
    })

quality_df = (
    pd.DataFrame(quality_by_lang)
    .sort_values(["rows", "mcn_lang"], ascending=[False, True])
    .reset_index(drop=True)
) if quality_by_lang else pd.DataFrame()
if not quality_df.empty:
    print("\nPer language (rows >= 1), most rows first:")
    display(quality_df.style.format({
        "exact": "{:.1%}", "fold": "{:.1%}", "compact": "{:.1%}",
        "sim": "{:.3f}", "sim_fold": "{:.3f}", "low_confidence": "{:.0%}",
    }))

In [ ]:
# Examples per language: the closest and the furthest pairs, so both typical
# agreement and the failure modes are visible
EXAMPLE_LANGS = 12
EXAMPLES_EACH = 4

example_rows = []
for lang in (quality_df["mcn_lang"].head(EXAMPLE_LANGS) if not quality_df.empty else []):
    qs = sorted(grouped[lang], key=lambda q: (-q.sim, q.mcn_name))
    picks = qs[:EXAMPLES_EACH] + [q for q in qs[-EXAMPLES_EACH:] if q not in qs[:EXAMPLES_EACH]]
    for q in picks:
        example_rows.append({
            "language": lang, "mcn": q.mcn_name, "nte_native": q.native,
            "nte_romanized": q.romanized, "sim": round(q.sim, 3), "confidence": q.confidence,
        })
display(pd.DataFrame(example_rows))

## Character substitutions: romanized → MCN

For each language, the differences between NTE's romanization and MCN's spelling are split into small edits: replacements, deletions and insertions of up to 3 characters, case-folded, with diacritics kept. The most frequent edits are what a readable display romanization would have to do on top of the technical one, e.g. `ž → zh`, `ʹ → (deleted)`.

In [ ]:
SUB_LANGS = 10
SUBS_EACH = 12
MAX_FRAGMENT = 3

sub_rows = []
for lang in (quality_df["mcn_lang"].head(SUB_LANGS) if not quality_df.empty else []):
    edits = collections.Counter()
    for q in grouped[lang]:
        a, b = norm_casefold(q.romanized), norm_casefold(q.mcn_name)
        for tag, i1, i2, j1, j2 in difflib.SequenceMatcher(None, a, b, autojunk=False).get_opcodes():
            if tag == "equal":
                continue
            src, dst = a[i1:i2], b[j1:j2]
            if len(src) <= MAX_FRAGMENT and len(dst) <= MAX_FRAGMENT:
                edits[(src or "∅", dst or "∅")] += 1
    for (src, dst), count in edits.most_common(SUBS_EACH):
        sub_rows.append({"language": lang, "romanized": src, "mcn": dst, "count": count})

display(pd.DataFrame(sub_rows))

In [ ]:
# Full-database collision audit:
# conservative key (casefold) vs diacritic-insensitive folded key

with get_conn() as conn:
    conn.execute("DROP TABLE IF EXISTS temp.lookup_collision_keys")

    # Materialize once so the expensive runtime normalization is not repeated
    # for every diagnostic query below.
    conn.execute("""
        CREATE TEMP TABLE lookup_collision_keys AS

        SELECT
            'geonames' AS source,
            CAST(geonameid AS TEXT) AS entity_id,
            name AS raw_name,
            nte_casefold(name) AS norm_key,
            nte_diacritic_fold(name) AS folded_key
        FROM geoname
        WHERE name IS NOT NULL AND name <> ''

        UNION ALL

        SELECT
            'geonames' AS source,
            CAST(geonameid AS TEXT) AS entity_id,
            alternate_name AS raw_name,
            nte_casefold(alternate_name) AS norm_key,
            nte_diacritic_fold(alternate_name) AS folded_key
        FROM alternate_name
        WHERE alternate_name IS NOT NULL AND alternate_name <> ''

        UNION ALL

        SELECT
            'wikidata' AS source,
            qid AS entity_id,
            name AS raw_name,
            nte_casefold(name) AS norm_key,
            nte_diacritic_fold(name) AS folded_key
        FROM wikidata_location_name
        WHERE name IS NOT NULL AND name <> ''
    """)

    conn.execute("""
        CREATE INDEX temp.idx_collision_norm
        ON lookup_collision_keys(source, norm_key, entity_id)
    """)

    conn.execute("""
        CREATE INDEX temp.idx_collision_folded
        ON lookup_collision_keys(source, folded_key, entity_id)
    """)

    # Overall collision rates for conservative and diacritic-insensitive keys.
    summary_df = pd.read_sql_query("""
        WITH
        norm_counts AS (
            SELECT
                source,
                norm_key AS key,
                COUNT(DISTINCT entity_id) AS entity_count
            FROM lookup_collision_keys
            WHERE norm_key <> ''
            GROUP BY source, norm_key
        ),
        folded_counts AS (
            SELECT
                source,
                folded_key AS key,
                COUNT(DISTINCT entity_id) AS entity_count
            FROM lookup_collision_keys
            WHERE folded_key <> ''
            GROUP BY source, folded_key
        ),
        summary AS (
            SELECT
                source,
                'casefold' AS policy,
                COUNT(*) AS distinct_keys,
                SUM(entity_count > 1) AS ambiguous_keys,
                MAX(entity_count) AS max_entities_per_key,
                AVG(entity_count) AS mean_entities_per_key
            FROM norm_counts
            GROUP BY source

            UNION ALL

            SELECT
                source,
                'diacritic_fold' AS policy,
                COUNT(*) AS distinct_keys,
                SUM(entity_count > 1) AS ambiguous_keys,
                MAX(entity_count) AS max_entities_per_key,
                AVG(entity_count) AS mean_entities_per_key
            FROM folded_counts
            GROUP BY source
        )
        SELECT
            *,
            CAST(ambiguous_keys AS REAL) / distinct_keys AS ambiguous_key_rate
        FROM summary
        ORDER BY source, policy
    """, conn)

    display(summary_df.style.format({
        "ambiguous_key_rate": "{:.3%}",
        "mean_entities_per_key": "{:.4f}",
    }))

    # Keys that become ambiguous specifically because diacritics were removed.
    # A "new collision" means:
    #   - the folded key maps to >1 entity
    #   - every conservative key contributing to it was individually unambiguous
    newly_ambiguous_df = pd.read_sql_query("""
        WITH norm_counts AS (
            SELECT
                source,
                norm_key,
                COUNT(DISTINCT entity_id) AS entity_count
            FROM lookup_collision_keys
            WHERE norm_key <> ''
            GROUP BY source, norm_key
        ),
        folded_groups AS (
            SELECT
                source,
                folded_key,
                COUNT(DISTINCT entity_id) AS folded_entity_count,
                COUNT(DISTINCT norm_key) AS conservative_key_count,
                MAX(
                    (
                        SELECT nc.entity_count
                        FROM norm_counts nc
                        WHERE nc.source = k.source
                          AND nc.norm_key = k.norm_key
                    )
                ) AS max_conservative_entity_count
            FROM lookup_collision_keys k
            WHERE folded_key <> ''
            GROUP BY source, folded_key
        )
        SELECT
            source,
            COUNT(*) AS newly_ambiguous_keys,
            MAX(folded_entity_count) AS max_entities_in_new_collision,
            AVG(folded_entity_count) AS mean_entities_in_new_collision
        FROM folded_groups
        WHERE folded_entity_count > 1
          AND max_conservative_entity_count = 1
        GROUP BY source
        ORDER BY source
    """, conn)

    print("Collisions introduced specifically by diacritic stripping:")
    display(newly_ambiguous_df)

    # Largest newly-created collisions, with example spellings and entity IDs.
    collision_examples_df = pd.read_sql_query("""
        WITH norm_counts AS (
            SELECT
                source,
                norm_key,
                COUNT(DISTINCT entity_id) AS entity_count
            FROM lookup_collision_keys
            WHERE norm_key <> ''
            GROUP BY source, norm_key
        ),
        folded_groups AS (
            SELECT
                k.source,
                k.folded_key,
                COUNT(DISTINCT k.entity_id) AS folded_entity_count,
                COUNT(DISTINCT k.norm_key) AS conservative_key_count,
                MAX(nc.entity_count) AS max_conservative_entity_count
            FROM lookup_collision_keys k
            JOIN norm_counts nc
              ON nc.source = k.source
             AND nc.norm_key = k.norm_key
            WHERE k.folded_key <> ''
            GROUP BY k.source, k.folded_key
        )
        SELECT
            fg.source,
            fg.folded_key,
            fg.folded_entity_count AS entity_count,
            fg.conservative_key_count,
            GROUP_CONCAT(DISTINCT k.raw_name) AS raw_names,
            GROUP_CONCAT(DISTINCT k.entity_id) AS entity_ids
        FROM folded_groups fg
        JOIN lookup_collision_keys k
          ON k.source = fg.source
         AND k.folded_key = fg.folded_key
        WHERE fg.folded_entity_count > 1
          AND fg.max_conservative_entity_count = 1
        GROUP BY
            fg.source,
            fg.folded_key,
            fg.folded_entity_count,
            fg.conservative_key_count
        ORDER BY fg.folded_entity_count DESC, fg.source, fg.folded_key
        LIMIT 100
    """, conn)

    print("Largest collisions introduced by diacritic stripping:")
    display(collision_examples_df)

    # How often stripping marks actually changes a stored name key.
    changed_df = pd.read_sql_query("""
        SELECT
            source,
            COUNT(*) AS name_rows,
            SUM(norm_key <> folded_key) AS changed_rows,
            CAST(SUM(norm_key <> folded_key) AS REAL) / COUNT(*) AS changed_rate
        FROM lookup_collision_keys
        GROUP BY source
        ORDER BY source
    """, conn)

    print("How much of the database is affected by diacritic stripping:")
    display(changed_df.style.format({"changed_rate": "{:.2%}"}))